# Cropchain Phase 2 — Scrum Planning
## Experiment 2: Sprint Execution — Development Tasks, QA Coverage, and UI/UX Design

**Project:** Cropchain: Intelligent Farm-to-Table Supply Chain Management System  
**Methodology:** Phase 2 — Agile Scrum  
**Author:** Shai Sunnuma  
**Course:** CS587 — Software Project Management  

---

## Experiment Overview

This notebook is your individual contribution to Phase 2 of the Cropchain final project. It picks up directly from the Team Lead's Experiment 1, which produced the Product Backlog and Sprint 1 Plan using the Scrum framework.

Your responsibility in this experiment is to simulate the **Sprint execution planning** stage of Scrum. Three specialized agents are activated here, each taking on a role from the Scrum development team:

- **Developer Agent** — translates Sprint 1 user stories into concrete technical tasks with role assignments and hour-level effort estimates
- **QA Engineer Agent** — produces test cases mapped to both user story IDs and Phase 1 requirement IDs, covering functional and non-functional testing
- **UI/UX Designer Agent** — contributes interface design details for all user-facing Sprint 1 stories

The Scrum Master agent coordinates all three, consistent with the role defined in Experiment 1.

### Why This Experiment Matters for the Project

The outputs you generate here are loaded directly into Teammate 2's Experiment 3. They will be used to build the DevOps pipeline plan, Sprint Review simulation, documentation plan, and the final Scrum vs Waterfall comparative analysis. **If your output files are missing or incomplete, Teammate 2's notebook will fail at startup.**

### Prerequisites — Confirm Before Running

The Team Lead must have already run Experiment 1. Verify that these two files exist before proceeding:
- `src/outputs/phase2/experiment_1/03_product_backlog.md`
- `src/outputs/phase2/experiment_1/04_sprint1_plan.md`

### Git Instructions
```bash
# Switch to the Phase 2 working branch and pull the latest
git checkout phase2
git pull origin phase2

# After finishing and cleaning up your notebook:
git add 05_Cropchain_Phase2_Experiment2_Dev_and_QA.ipynb
git commit -m "Phase 2 Exp 2: [Your Name] - Developer tasks, QA coverage, UI/UX design"
git push origin phase2
```

## Section 1: Environment Setup and Artifact Loading

The environment configuration is identical to Phase 1 and the Team Lead's Experiment 1 notebook. The same model (`gpt-4o-mini`), the same `TEAM_LLM_CONFIG`, and the same `.env` file are used across all team notebooks to ensure consistency and reproducibility.

This section also loads the two upstream artifacts produced by the Team Lead. These files are injected as context into every agent prompt in this notebook so that the Developer, QA, and Designer agents all work from the actual Sprint 1 scope — not from a generic or invented starting point.

If either file is missing, the cell will raise a `FileNotFoundError` with a message telling you what to do.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import autogen

CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

load_dotenv(BASE_DIR / ".env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")

MODEL_NAME = "gpt-4o-mini"
TEAM_LLM_CONFIG = {
    "config_list": [{"model": MODEL_NAME, "api_key": OPENAI_API_KEY}],
    "temperature": 0.2,
    "timeout": 120,
}

PROJECT_NAME = "Cropchain: Intelligent Farm-to-Table Supply Chain Management System"

# Directories
PHASE2_EXP1_DIR  = BASE_DIR / "src" / "outputs" / "phase2" / "experiment_1"
OUTPUT_DIR       = BASE_DIR / "src" / "outputs" / "phase2" / "experiment_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load upstream artifacts from Team Lead's Experiment 1
BACKLOG_FILE      = PHASE2_EXP1_DIR / "03_product_backlog.md"
SPRINT1_PLAN_FILE = PHASE2_EXP1_DIR / "04_sprint1_plan.md"

for f in [BACKLOG_FILE, SPRINT1_PLAN_FILE]:
    if not f.exists():
        raise FileNotFoundError(
            f"Required Team Lead file missing: {f}\n"
            "Ask the Team Lead to run Experiment 1 and push to phase2 branch first."
        )

product_backlog_text = BACKLOG_FILE.read_text(encoding="utf-8")
sprint1_plan_text    = SPRINT1_PLAN_FILE.read_text(encoding="utf-8")

print("Environment loaded successfully.")
print("Model:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR)
print("Product Backlog loaded:", bool(product_backlog_text.strip()))
print("Sprint 1 Plan loaded:",   bool(sprint1_plan_text.strip()))

## Section 2: Agent Definitions

Four agents are used in this experiment. The **Scrum Master** coordinates and initiates all conversations. The three **specialist agents** — Developer, QA Engineer, and UI/UX Designer — each respond with structured, Cropchain-specific outputs.

The Scrum Master's system message is provided below as a working reference example. Study it carefully — pay attention to how it defines the role, lists responsibilities, and ends with a formatting instruction. Your three agent system messages should follow the same level of detail and specificity.

**Your task:** Write the `system_message` for the Developer Agent, QA Engineer Agent, and Designer Agent. Each agent must know:
- What their specific Scrum role is on the Cropchain team
- Exactly what outputs they are expected to produce (numbered, specific)
- Any productivity assumptions that apply to them
- That their output must be Cropchain-specific — not generic software planning
- The shared technology stack: React.js (frontend), Node.js + Express (backend), PostgreSQL (database)

> **Reference:** Scrum team roles — https://www.atlassian.com/agile/scrum/roles

In [ ]:
# Scrum Master — provided as a complete reference. Do not modify.
scrum_master_agent = autogen.ConversableAgent(
    name="Scrum_Master",
    system_message="""
You are the Scrum Master for the Cropchain Scrum project.
In this experiment you coordinate the development team.
Route tasks to the correct team members and ensure outputs stay consistent
with the Sprint 1 Plan and Product Backlog.
Keep outputs structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

developer_agent = autogen.ConversableAgent(
    name="Developer_Agent",
    system_message="""
You are a Full-Stack Developer on the Cropchain Scrum team, working on the Cropchain:
Intelligent Farm-to-Table Supply Chain Management System.

Your role in this sprint is to translate each Sprint 1 user story into a concrete,
implementation-ready technical task breakdown. Your output is used directly by the
development team and must be specific to Cropchain — not generic software planning.

Technology stack:
- Frontend: React.js
- Backend: Node.js + Express
- Database: PostgreSQL

For every Sprint 1 user story, produce the following:
1. A breakdown of 3–5 concrete technical tasks required to implement that story.
2. A role assignment for each task: Backend / Frontend / Database / AI Service / DevOps.
3. An hour-level effort estimate for each task (assume 6 productive hours per developer per day).
4. Technical dependencies between tasks (e.g., "Task B depends on Task A — database schema must exist first").
5. A note for any task that addresses a non-functional requirement: performance (<2s response),
   scalability (10,000 concurrent users), uptime (99.9%), or GDPR data privacy.
6. One review task and one rework/buffer task per user story.

Productivity assumption: each developer works 6 productive hours per day.
All outputs must reference Cropchain domain concepts such as farmer registration,
crop listings, buyer ordering, and alert notifications.
Make the output structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

qa_engineer_agent = autogen.ConversableAgent(
    name="QA_Engineer",
    system_message="""
You are the QA / Test Engineer on the Cropchain Scrum team, working on the Cropchain:
Intelligent Farm-to-Table Supply Chain Management System.

Your role is to produce a comprehensive test plan for Sprint 1. Your output is used directly
by the QA team and must be specific to Cropchain — not generic test planning.

For every Sprint 1 user story, produce the following:
1. 2–3 test cases per user story.
2. Each test case must be mapped to both its user story ID (US-XXX) and the
   corresponding Phase 1 requirement ID (REQ-XXX).
3. The test type for each case: Unit / Integration / System / Acceptance.
4. Clear test steps and expected results for each test case.
5. Explicit coverage of Cropchain-specific features where applicable:
   yield prediction, fair pricing recommendations, shipment alerts,
   crop traceability, and inventory visibility.
6. A confirmation that each story's acceptance criteria are testable as written.
7. A summary at the end showing: total test case count and total estimated effort
   (productivity assumption: 2 test cases completed per QA engineer per day).

Requirement IDs to reference (REQ-001 through REQ-017) come from the Phase 1
Requirements Engineering output. Map each test case to the closest matching REQ.
Make the output structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

designer_agent = autogen.ConversableAgent(
    name="Designer_Agent",
    system_message="""
You are the UI/UX Designer on the Cropchain Scrum team, working on the Cropchain:
Intelligent Farm-to-Table Supply Chain Management System.

Your role is to produce interface design plans for all user-facing Sprint 1 stories.
Your output is used by the frontend developers and must be specific to Cropchain
and its three primary user types: Farmer, Restaurant Buyer, and Grocery Buyer.

For every UI-relevant Sprint 1 user story, produce the following:
1. A plain-text wireframe description of the key screen(s) — describe layout regions,
   key components, and interaction flows without using images or diagrams.
2. Accessibility considerations specific to that story (e.g., screen reader support,
   color contrast, keyboard navigation for form fields).
3. UX pain points the design addresses for Cropchain users (e.g., farmers with
   limited tech literacy, buyers on mobile devices in fast-paced environments).
4. A design effort estimate in hours (assume 4 productive design hours per day).
5. Explicit confirmation that the design is mobile-responsive (REQ-013: accessible
   via web browsers on desktop and mobile devices).

Skip any Sprint 1 story that is purely backend with no user-facing interface.
State clearly which stories are skipped and why.
Make the output structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

print("All Experiment 2 agents created successfully.")

## Section 3: Prompt Design

Each prompt is sent by the Scrum Master to one specialist agent. All three prompts inject `product_backlog_text` and `sprint1_plan_text` as context so the agents work from the actual Cropchain Sprint 1 scope rather than generating generic content.

**Your task:** Write all three prompts below as f-strings. A well-constructed prompt for this project:
- Opens by establishing what has already been completed (Sprint 1 planning is done)
- Includes labeled context sections using the upstream variables already loaded
- Gives clear, numbered output requirements specific to the agent's role
- References Cropchain domain specifics (actors, features, non-functional requirements)
- Ends with a formatting instruction: structured and presentation-ready

Study the prompts in the Team Lead's Experiment 1 notebook and the Phase 1 notebooks as reference before writing yours.

In [ ]:
sm_to_developer_prompt = f"""Sprint 1 planning for the Cropchain project is complete. The Product Backlog and Sprint 1 Plan have been finalized by the Scrum Master and are provided below for your reference.

--- PRODUCT BACKLOG ---
{product_backlog_text}

--- SPRINT 1 PLAN ---
{sprint1_plan_text}

Your task is to produce a technical task breakdown for every user story committed to Sprint 1 (US-001, US-002, US-003, US-004, US-005, US-006, US-012).

For each story, provide:
1. 3–5 concrete technical tasks required to implement that story.
2. A role label for each task: Backend / Frontend / Database / AI Service / DevOps.
3. An hour-level effort estimate per task (6 productive hours per developer per day).
4. Technical dependencies between tasks (state which task must complete before another starts).
5. A non-functional requirement flag where applicable — note which tasks address: performance (<2s response time, REQ-014), scalability (10,000 users, REQ-015), uptime (99.9%, REQ-016), or GDPR compliance (REQ-017).
6. One code review task and one rework/buffer task per story.

Technology stack: React.js (frontend), Node.js + Express (backend), PostgreSQL (database).
All tasks must reference Cropchain domain concepts — farmer registration, crop listing, buyer ordering, and alert notifications — not generic software boilerplate.

Make the output structured and presentation-ready.""".strip()

sm_to_qa_prompt = f"""Sprint 1 planning for the Cropchain project is complete. The Product Backlog and Sprint 1 Plan have been finalized by the Scrum Master and are provided below for your reference.

--- PRODUCT BACKLOG ---
{product_backlog_text}

--- SPRINT 1 PLAN ---
{sprint1_plan_text}

Your task is to produce a test plan covering every user story committed to Sprint 1 (US-001, US-002, US-003, US-004, US-005, US-006, US-012).

For each story, provide:
1. 2–3 test cases.
2. Each test case mapped to both its user story ID (US-XXX) and the corresponding Phase 1 requirement ID (REQ-XXX). Use REQ-001 through REQ-017 from the Cropchain requirements baseline.
3. The test type: Unit / Integration / System / Acceptance.
4. Step-by-step test steps and clear expected results.
5. Explicit coverage of these Cropchain-specific features where applicable: farmer account creation, crop listing visibility, buyer order placement, and alert delivery for delays and shortages.
6. Confirmation that each story's acceptance criteria (as listed in the backlog) are testable as written.

At the end, provide a summary with:
- Total number of test cases produced.
- Total estimated QA effort in days (productivity assumption: 2 test cases per QA engineer per day).

Make the output structured and presentation-ready.""".strip()

sm_to_designer_prompt = f"""Sprint 1 planning for the Cropchain project is complete. The Product Backlog and Sprint 1 Plan have been finalized by the Scrum Master and are provided below for your reference.

--- PRODUCT BACKLOG ---
{product_backlog_text}

--- SPRINT 1 PLAN ---
{sprint1_plan_text}

Your task is to produce UI/UX design plans for all user-facing stories committed to Sprint 1 (US-001, US-002, US-003, US-004, US-005, US-006, US-012).

For each UI-relevant story, provide:
1. A plain-text wireframe description of the key screen(s) — describe layout regions, primary components (forms, buttons, tables, navigation), and the main interaction flow. No images or diagrams.
2. Accessibility considerations specific to that story (e.g., form label associations, color contrast ratios, keyboard-tab ordering, screen reader announcements).
3. UX pain points the design addresses for Cropchain's primary users: Farmer, Restaurant Buyer, and Grocery Buyer (consider varying tech literacy levels and mobile-first contexts).
4. A design effort estimate in hours (assume 4 productive design hours per day).
5. An explicit confirmation that the design is mobile-responsive and meets REQ-013 (accessible via web browsers on desktop and mobile devices).

If a Sprint 1 story is purely backend with no user-facing interface, skip it and state clearly which story was skipped and why.

Make the output structured and presentation-ready.""".strip()

print("Prompts prepared.")
print("Developer prompt preview (first 300 chars):")
print(sm_to_developer_prompt[:300] + "\n...")

## Section 4: Output Persistence Helpers

These helper functions are shared across all Phase 2 notebooks. They save agent outputs as markdown files and raw chat histories to the experiment output directory. The three markdown files produced here are the artifacts that Teammate 2 will load in Experiment 3.

In [ ]:
def save_text(filename: str, text: str):
    """Save a string to a file in the experiment output directory."""
    file_path = OUTPUT_DIR / filename
    file_path.write_text(text, encoding="utf-8")
    print(f"Saved: {file_path}")

def save_chat_history(filename: str, chat_result):
    """Save the full AutoGen chat history to a readable text file."""
    file_path = OUTPUT_DIR / filename
    if hasattr(chat_result, "chat_history"):
        lines = []
        for item in chat_result.chat_history:
            role = item.get("name", item.get("role", "Unknown"))
            content = item.get("content", "")
            lines.append(f"{role}:\n{content}\n")
        file_path.write_text(("\n" + ("-" * 80) + "\n").join(lines), encoding="utf-8")
        print(f"Saved chat history: {file_path}")
    else:
        print("No chat_history found on chat result.")

def extract_last_message(chat_result) -> str:
    """Extract the final agent response from a completed chat result."""
    if hasattr(chat_result, "chat_history") and chat_result.chat_history:
        return chat_result.chat_history[-1].get("content", "")
    return ""

print("Output helpers defined.")

## Section 5: Run Experiment 2

This section executes the three-step agent workflow sequentially:

1. **Scrum Master → Developer Agent** — technical task breakdown for all Sprint 1 stories
2. **Scrum Master → QA Engineer** — test cases mapped to story and requirement IDs
3. **Scrum Master → Designer Agent** — UI/UX design notes for interface-facing stories

Each conversation uses `max_turns=1` — one prompt, one structured response. This is the same pattern used throughout the project.

In [ ]:
def run_phase2_experiment_2():

    print("=" * 80)
    print("STEP 1: Scrum Master → Developer Agent")
    print("=" * 80)
    sm_to_dev_result = scrum_master_agent.initiate_chat(
        recipient=developer_agent,
        message=sm_to_developer_prompt,
        max_turns=1,
    )

    print("\n" + "=" * 80)
    print("STEP 2: Scrum Master → QA Engineer")
    print("=" * 80)
    sm_to_qa_result = scrum_master_agent.initiate_chat(
        recipient=qa_engineer_agent,
        message=sm_to_qa_prompt,
        max_turns=1,
    )

    print("\n" + "=" * 80)
    print("STEP 3: Scrum Master → Designer Agent")
    print("=" * 80)
    sm_to_designer_result = scrum_master_agent.initiate_chat(
        recipient=designer_agent,
        message=sm_to_designer_prompt,
        max_turns=1,
    )

    return sm_to_dev_result, sm_to_qa_result, sm_to_designer_result

sm_to_dev_result, sm_to_qa_result, sm_to_designer_result = run_phase2_experiment_2()

## Section 6: Save Experiment 2 Artifacts

The outputs from each agent are saved as markdown files to `src/outputs/phase2/experiment_2/`. These three files are required inputs for Teammate 2's Experiment 3 — without them, Experiment 3 will fail to load at startup. Once this section runs without errors, notify Teammate 2 that their inputs are ready.

In [ ]:
save_chat_history("01_sm_to_developer_chat.txt", sm_to_dev_result)
save_chat_history("02_sm_to_qa_chat.txt",        sm_to_qa_result)
save_chat_history("03_sm_to_designer_chat.txt",   sm_to_designer_result)

dev_output      = extract_last_message(sm_to_dev_result)
qa_output       = extract_last_message(sm_to_qa_result)
designer_output = extract_last_message(sm_to_designer_result)

if dev_output:      save_text("04_developer_tasks.md",  dev_output)
if qa_output:       save_text("05_qa_test_cases.md",     qa_output)
if designer_output: save_text("06_design_notes.md",      designer_output)

print("\nPhase 2 Experiment 2 artifacts saved.")

## Section 7: Artifact Validation

This cell confirms that all three output files were successfully written. All three must show `FOUND` before you push to the branch or notify Teammate 2.

In [ ]:
required_files = [
    OUTPUT_DIR / "04_developer_tasks.md",
    OUTPUT_DIR / "05_qa_test_cases.md",
    OUTPUT_DIR / "06_design_notes.md",
]

all_found = True
for f in required_files:
    status = "FOUND" if f.exists() else "MISSING"
    print(f"{f.name}: {status}")
    if status == "MISSING":
        all_found = False

if all_found:
    print("\nAll artifacts validated. Notify Teammate 2 that Experiment 3 inputs are ready.")
else:
    print("\nWARNING: Fix missing files before notifying Teammate 2.")

## Section 8: Display Generated Outputs

This section renders all three generated markdown files inline for review. Read through each output carefully before completing your analysis in Section 9. Your evaluation must reflect what the agents actually produced — not what you expected them to produce.

In [ ]:
from IPython.display import Markdown, display

for filename, label in [
    ("04_developer_tasks.md",  "DEVELOPER TECHNICAL TASK BREAKDOWN"),
    ("05_qa_test_cases.md",    "QA TEST CASES"),
    ("06_design_notes.md",     "UI/UX DESIGN NOTES"),
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print("=" * 60)
        print(label)
        print("=" * 60)
        display(Markdown(path.read_text(encoding="utf-8")))

## Section 9: Experiment Analysis and Evaluation

**This entire section must be written by you in your own words after running the experiment. Delete all placeholder text before submitting.**

---

### 9.1 What This Experiment Produced

*[Write 3-5 sentences summarizing what the three agents generated. Be specific — mention actual story IDs, task roles, or test case types that appeared in your output. Do not write a generic description.]*

---

### 9.2 Developer Agent — Accuracy Assessment

*[Evaluate the Developer Agent's technical task breakdown. Did the tasks align with the actual Sprint 1 stories? Were hour estimates reasonable for a 6-hour productive day? Were the right roles (Backend, Frontend, Database, AI Service) assigned? Did the agent correctly reference the PostgreSQL and React.js stack from Phase 1? Point to specific examples from the output.]*

---

### 9.3 QA Engineer — Coverage Assessment

*[Evaluate the QA Engineer's output. Are all Sprint 1 stories covered? Do the test cases correctly map to both story IDs (US-XXX) and requirement IDs (REQ-XXX)? Are Cropchain-specific features — yield prediction, fair pricing, traceability, alerts — tested? Was the effort estimate (2 test cases/day) applied correctly? What is missing or incomplete?]*

---

### 9.4 UI/UX Designer — Design Quality Assessment

*[Evaluate the Designer Agent's output. Are the wireframe descriptions meaningful and specific to Cropchain's users (Farmer, Buyer, Admin)? Are accessibility considerations genuine, not generic? Is mobile responsiveness addressed (REQ-013)? Which stories were skipped as backend-only, and was that correct?]*

---

### 9.5 Coherence with Phase 1 and Experiment 1

*[Discuss whether this experiment's outputs are consistent with the broader project. Do the developer tasks use the same technology stack defined in Phase 1 Experiment 2 (PostgreSQL, React.js, Node.js)? Do the QA test cases reference the same requirement IDs (REQ-001 to REQ-017) established in Phase 1 Experiment 1? Does the Sprint 1 scope in your outputs match what the Team Lead's Scrum Master committed to?]*

---

### 9.6 Limitations

*[Acknowledge at least 3 real limitations of this experiment. Consider: are hour estimates verified or assumed? Can plain-text wireframe descriptions replace actual design mockups? What would a real sprint execution reveal that this simulation cannot? Are the test cases executable as written, or do they require more detail?]*

---

### 9.7 Conclusion

*[Write 3-4 sentences concluding this experiment. State what was achieved, why it matters for the Cropchain Phase 2 project, and what Teammate 2 can now do with the outputs you produced.]*